In [3]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


In [4]:
# Sélection du scope de données à traiter (sources, défauts, diamètres, positions, charges, canaux)
selection = data.selectionner(
    sources=("12kDE", "12kFE", "Normal"),
    defauts=("IR", "B", "OR"),
    diametres=(7, 14, 21), positions_or=("6",),
    charges=(0, 1, 2, 3),
    canaux={"12kDE": ("DE",), 
            "12kFE": ("FE",), 
            "Normal": ("DE","FE")},
)

In [5]:
# Importation des données sélectionnées et ajout des indicateurs temporels et spectraux (ajout de ceux de Mourad)
table = features.construire_table_SpecMourad(selection, data.charger)

In [20]:
# Preparation pour le modeling + modeling
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lazypredict import LazyClassifier
for contexte in ["DE→FE"]:
    for target in ["classe"]:
        for indics in ["Spectre Mourad"]:
            if indics == "Spectre Robin":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_SPECTRAUX
            elif indics == "Spectre Mourad":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_Mourad

            if contexte == "DE→DE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling(table, COLONNES_INDICATEURS, target=target)
            elif contexte == "DE→FE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_DE_FE(table, COLONNES_INDICATEURS, canal_train="DE", canal_test="FE", target=target)
            
            print("################### Contexte #######################")
            print("####################################################")
            print(f"######## Train→Test : {contexte}")
            print(f"######## target : {target}")
            print(f"######## Features : Temporels + {indics}")
            display(X_train.head())

            X_train, y_train, X_test, y_test = features.equilibrer(X_train, y_train, X_test, y_test, type="train")
            display(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}))
            
            print("\n#=====================================#")
            print("#====== LogReg VS Random Forest ======#")
            print("#=====================================#")

            # modeles = {
            #     "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
            #     "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
            # }
            # tableau = models.entrainer_comparer(modeles, X_train, y_train, X_test, y_test, report=False, dispMatrice=False)
            # tableau

            clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
            models,predictions = clf.fit(X_train, X_test, y_train, y_test)
            models
            
            print("\n")

################### Contexte #######################
####################################################
######## Train→Test : DE→FE
######## target : classe
######## Features : Temporels + Spectre Mourad


,RMS,Pic,Crete,Kurtosis,RMS_bande,Env_BPFI,Env_BPFO,Env_BSF,Env_FTF
0,-0.801325,-0.746152,-0.643027,-0.667171,-0.877362,-0.667873,-0.495991,-0.341482,2.760346
1,-0.775023,-0.777891,-1.043185,-0.666541,-0.867098,-0.733118,-0.534447,-0.431474,2.642486
2,-0.799978,-0.762380,-0.803960,-0.677066,-0.869663,-0.655710,-0.475214,0.570954,3.810059
3,-0.802151,-0.763673,-0.806956,-0.655665,-0.867849,-0.581123,-0.480116,-0.702944,2.984571
4,-0.810091,-0.779327,-0.925166,-0.671120,-0.873104,-0.672491,-0.505730,-0.319380,3.253046


,train,test
classe,,
B007,200,200
B014,200,200
B021,200,200
IR007,200,200
IR014,200,200
IR021,200,200
OR007,200,200
OR014,200,50
OR021,200,50



#=====================================#
#====== LogReg VS Random Forest ======#
#=====================================#




In [ ]:
models["scoreTOT"] = (models["Accuracy"] + models["Balanced Accuracy"] + models["ROC AUC"])
models["scoreTimed"] = (models["Accuracy"] * models["Balanced Accuracy"] * models["ROC AUC"]) / (models["Time Taken"] + 1e-6)
models = models.sort_values(by="scoreTimed", ascending=False)
models 

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken,scoreTOT,scoreTimed
Model,,,,,,,,,
GaussianNB,0.490027,0.389377,0.872692,0.454329,0.458778,0.490027,0.112893,1.752096,1.474960
NearestCentroid,0.449229,0.412224,0.893383,0.472826,0.572241,0.449229,0.134708,1.754836,1.228123
DecisionTreeClassifier,0.533092,0.335000,0.745642,0.491227,0.473943,0.533092,0.125988,1.613733,1.056924
BaggingClassifier,0.520399,0.369378,0.824595,0.479314,0.475743,0.520399,0.221121,1.714373,0.716830
LinearDiscriminantAnalysis,0.411151,0.275500,0.768033,0.287124,0.220729,0.411151,0.121587,1.454684,0.715507
KNeighborsClassifier,0.503173,0.402802,0.771404,0.494794,0.536110,0.503173,0.231625,1.677379,0.674999
BernoulliNB,0.301904,0.311161,0.814663,0.282745,0.411020,0.301904,0.121596,1.427728,0.629377
ExtraTreesClassifier,0.507706,0.320792,0.879785,0.464218,0.464363,0.507706,0.403596,1.708283,0.355029
LogisticRegression,0.398005,0.311528,0.838672,0.400393,0.485537,0.398005,0.292992,1.548205,0.354912


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken,scoreTOT,scoreTimed
Model,,,,,,,,,
GaussianNB,0.490027,0.389377,0.872692,0.454329,0.458778,0.490027,0.112893,1.752096,1.474960
NearestCentroid,0.449229,0.412224,0.893383,0.472826,0.572241,0.449229,0.134708,1.754836,1.228123
DecisionTreeClassifier,0.533092,0.335000,0.745642,0.491227,0.473943,0.533092,0.125988,1.613733,1.056924
BaggingClassifier,0.520399,0.369378,0.824595,0.479314,0.475743,0.520399,0.221121,1.714373,0.716830
LinearDiscriminantAnalysis,0.411151,0.275500,0.768033,0.287124,0.220729,0.411151,0.121587,1.454684,0.715507
KNeighborsClassifier,0.503173,0.402802,0.771404,0.494794,0.536110,0.503173,0.231625,1.677379,0.674999
BernoulliNB,0.301904,0.311161,0.814663,0.282745,0.411020,0.301904,0.121596,1.427728,0.629377
ExtraTreesClassifier,0.507706,0.320792,0.879785,0.464218,0.464363,0.507706,0.403596,1.708283,0.355029
LogisticRegression,0.398005,0.311528,0.838672,0.400393,0.485537,0.398005,0.292992,1.548205,0.354912


In [7]:
# Preparation pour le modeling + modeling
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lazypredict import LazyClassifier
for contexte in ["DE→DE", "DE→FE"]:
    for target in ["defaut", "classe"]:
        for indics in ["Spectre Robin", "Spectre Mourad"]:
            if indics == "Spectre Robin":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_SPECTRAUX
            elif indics == "Spectre Mourad":
                COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS + indicateurs.COLONNES_Mourad

            if contexte == "DE→DE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling(table, COLONNES_INDICATEURS, target=target)
            elif contexte == "DE→FE":
                X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_DE_FE(table, COLONNES_INDICATEURS, canal_train="DE", canal_test="FE", target=target)
            
            print("################### Contexte #######################")
            print("####################################################")
            print(f"######## Train→Test : {contexte}")
            print(f"######## target : {target}")
            print(f"######## Features : Temporels + {indics}")
            display(X_train.head())

            X_train, y_train, X_test, y_test = features.equilibrer(X_train, y_train, X_test, y_test, type="train")
            display(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}))
            
            print("\n#=====================================#")
            print("#====== LogReg VS Random Forest ======#")
            print("#=====================================#")

            # modeles = {
            #     "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
            #     "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
            # }
            # tableau = models.entrainer_comparer(modeles, X_train, y_train, X_test, y_test, report=False, dispMatrice=False)
            # tableau



            
            print("\n")

################### Contexte #######################
####################################################
######## Train→Test : DE→DE
######## target : defaut
######## Features : Temporels + Spectre Robin


,RMS,Pic,Crete,Kurtosis,amp_BPFI_1,amp_BPFI_2,amp_BPFI_3,amp_BPFO_1,amp_BPFO_2,amp_BPFO_3,amp_BSF_1,amp_BSF_2,amp_BSF_3,amp_FTF_1,amp_FTF_2,amp_FTF_3,ratio_BPFI,ratio_BPFO,ratio_BSF,ratio_FTF
0,-0.819789,-0.796665,-0.716648,-0.690472,-0.792758,-0.641495,-0.664345,-0.536316,-0.554386,-0.624997,-0.689091,-0.835070,-0.665685,-0.643176,-0.566620,-0.554309,-0.742098,0.035625,-1.267802,1.364837
1,-0.793569,-0.831419,-1.176828,-0.689737,-0.790182,-0.641778,-0.687815,-0.540987,-0.541357,-0.617549,-0.670047,-0.791885,-0.653440,-0.671320,-0.565896,-0.554114,-0.580153,-0.379989,0.269346,0.810357
2,-0.818446,-0.814435,-0.901720,-0.702024,-0.800876,-0.646953,-0.697645,-0.538167,-0.552702,-0.629825,-0.679434,-0.819042,-0.584961,-0.727938,-0.536224,-0.518355,-0.801923,0.857528,0.513466,-0.185470
3,-0.820612,-0.815851,-0.905165,-0.677041,-0.791065,-0.633076,-0.653528,-0.534296,-0.548405,-0.617550,-0.672599,-0.760224,-0.652321,-0.682637,-0.545070,-0.528902,-0.657738,0.278181,-0.025555,0.448067
4,-0.828528,-0.832992,-1.041106,-0.695082,-0.797083,-0.637815,-0.629594,-0.545539,-0.541514,-0.614920,-0.675337,-0.887336,-0.672670,-0.641588,-0.540057,-0.508838,-0.863401,-0.846443,-0.048548,1.678588


,train,test
defaut,,
B,702,300
IR,702,300
OR,702,201
Sain,702,404



#=====================================#
#====== LogReg VS Random Forest ======#
#=====================================#


################### Contexte #######################
####################################################
######## Train→Test : DE→DE
######## target : defaut
######## Features : Temporels + Spectre Mourad


,RMS,Pic,Crete,Kurtosis,RMS_bande,Env_BPFI,Env_BPFO,Env_BSF,Env_FTF
0,-0.819789,-0.796665,-0.716648,-0.690472,-0.853796,-0.713142,-0.560211,-0.540465,1.474922
1,-0.793569,-0.831419,-1.176828,-0.689737,-0.843432,-0.777772,-0.598350,-0.587745,1.399816
2,-0.818446,-0.814435,-0.901720,-0.702024,-0.846022,-0.701093,-0.539606,-0.061087,2.143847
3,-0.820612,-0.815851,-0.905165,-0.677041,-0.844190,-0.627209,-0.544467,-0.730371,1.617809
4,-0.828528,-0.832992,-1.041106,-0.695082,-0.849496,-0.717716,-0.569869,-0.528853,1.788893


,train,test
defaut,,
B,702,300
IR,702,300
OR,702,201
Sain,702,404


... (15 outputs left out)


#=====================================#
#====== LogReg VS Random Forest ======#
#=====================================#


################### Contexte #######################
####################################################
######## Train→Test : DE→FE
######## target : classe
######## Features : Temporels + Spectre Mourad


,RMS,Pic,Crete,Kurtosis,RMS_bande,Env_BPFI,Env_BPFO,Env_BSF,Env_FTF
0,-0.801325,-0.746152,-0.643027,-0.667171,-0.877362,-0.667873,-0.495991,-0.341482,2.760346
1,-0.775023,-0.777891,-1.043185,-0.666541,-0.867098,-0.733118,-0.534447,-0.431474,2.642486
2,-0.799978,-0.762380,-0.803960,-0.677066,-0.869663,-0.655710,-0.475214,0.570954,3.810059
3,-0.802151,-0.763673,-0.806956,-0.655665,-0.867849,-0.581123,-0.480116,-0.702944,2.984571
4,-0.810091,-0.779327,-0.925166,-0.671120,-0.873104,-0.672491,-0.505730,-0.319380,3.253046


,train,test
classe,,
B007,200,200
B014,200,200
B021,200,200
IR007,200,200
IR014,200,200
IR021,200,200
OR007,200,200
OR014,200,50
OR021,200,50



#=====================================#
#====== LogReg VS Random Forest ======#
#=====================================#


